# 67 - Anatomical Gating Diagnostics for Strict UltraTimTrack

This notebook diagnoses the transient FL/ANG failures seen in `test_23june_2` and `test23_june_3` without changing production code.

The goal is not blind smoothing. The tested variants only reject or down-weight image-derived measurements when they are inconsistent with the current tracked state or with plausible frame-to-frame anatomy.

Questions addressed here:

- Are the FL spikes linked to accepted deep-aponeurosis jumps?
- Is there a maximum fascicle angle that could explain steep-angle failures?
- What are the bold red and thin yellow lines in the strict-run overlays?
- Do innovation gates, jump gates, component-specific R scales, and candidate persistence improve the problematic windows while preserving true motion elsewhere?


## 1. Setup

Outputs are written to `results/notebook67_anatomical_gating_diagnostics`.


In [1]:
from __future__ import annotations

import copy
import csv
import json
import math
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Mapping, Sequence

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import loadmat

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import ultrasound_tracker.roi as roi
import ultrasound_tracker.utils as ut
from scripts.run_strict_ultratimtrack_video import (
    apply_aponeurosis_maxangle_overrides,
    geofeature_alpha,
    geofeature_measurement_lines,
    update_parms_from_rois,
)
from ultrasound_tracker.matlab_timtrack import run_timtrack_geofeatures_from_video
from ultrasound_tracker.strict_fascicle_seed import FascicleSeedScoringConfig
from ultrasound_tracker.ultratrack_klt import (
    UltraTrackKLTConfig,
    apply_affine_1b,
    detect_min_eigen_like,
    estimate_affine_matlab_coords,
    filter_points_by_mask,
    tracking_masks_from_geofeature,
    track_points,
)
from ultrasound_tracker.ultratimtrack_aponeurosis import (
    aponeurosis_state_from_lines,
    lines_from_aponeurosis_state,
)
from ultrasound_tracker.ultratimtrack_matlab_2state import (
    MatlabTwoStateKalmanConfig,
    matlab_scalar_kalman_update,
    run_matlab_2state_kalman,
)

OUT_ROOT = ROOT / 'results' / 'notebook67_anatomical_gating_diagnostics'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
UTT_EXPORT = Path('/Users/grosbedou/Documents/GitHub/UltraTimTrack/UTT_numeric_export.mat')
assert UTT_EXPORT.exists(), UTT_EXPORT

DATASETS = {
    'test_23june_2': {
        'video': ROOT / 'data' / 'raw' / 'test_23june_2.AVI',
        'roi': ROOT / 'data' / 'rois' / 'test_23june_2_rois.json',
        'run_dir': ROOT / 'results' / 'strict_ultratimtrack_runs' / 'test_23june_2',
        'problem_frames': list(range(102, 111)),
        'overlay_frames': [102, 106, 107, 108, 110],
    },
    'test23_june_3': {
        'video': ROOT / 'data' / 'raw' / 'test23_june_3.AVI',
        'roi': ROOT / 'data' / 'rois' / 'test23_june_3_rois.json',
        'run_dir': ROOT / 'results' / 'strict_ultratimtrack_runs' / 'test23_june_3',
        'problem_frames': list(range(88, 92)),
        'overlay_frames': [88, 89, 90, 91, 95, 99],
    },
}

for name, spec in DATASETS.items():
    assert spec['video'].exists(), spec['video']
    assert spec['roi'].exists(), spec['roi']
    assert (spec['run_dir'] / f'{name}_strict_results.npz').exists(), name

print('root:', ROOT)
print('output:', OUT_ROOT)


root: /Users/grosbedou/PycharmProjects/NDORMS
output: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics


## 2. Current Line Meaning and Angle Limits

In `run_strict_ultratimtrack_video.py`, the thin yellow/orange line is `klt_prior_segments`: the fascicle seed propagated by KLT image motion before the final two-state Kalman correction. The bold red line is normally the final Kalman fascicle segment. If `--annotate-kalman-comparison` is enabled, red becomes the fixed-R Kalman result and blue is the selected adaptive Kalman result.

Angle limits are separate:

- TimTrack fascicle Hough detection uses the MATLAB/UTT `parms['fas']['range']` value. In this export it is broad: 5 to 60 degrees.
- The autonomous first-frame seed search is narrower by default: 14 to 24 degrees.
- Aponeurosis fitting has its own `maxangle`; the existing strict runs used 20 degrees for both superficial and deep aponeuroses.

So a steep fascicle is not blocked by the Hough range, but the initial seed and the downstream KLT/Kalman prior can still bias the result toward the initial moderate-angle fascicle unless we add candidate persistence or expose seed-angle settings.


In [2]:
mat_root = loadmat(UTT_EXPORT, simplify_cells=True)['UTT_numeric_export']
seed_cfg = FascicleSeedScoringConfig()

rows = []
for name, spec in DATASETS.items():
    metadata = json.loads((spec['run_dir'] / f'{name}_strict_metadata.json').read_text())
    cap = cv2.VideoCapture(str(spec['video']))
    ok, frame0 = cap.read()
    cap.release()
    assert ok
    gray0 = cv2.cvtColor(frame0, cv2.COLOR_BGR2GRAY) if frame0.ndim == 3 else frame0
    parms = copy.deepcopy(mat_root['parms'])
    parms = update_parms_from_rois(parms, roi.load_rois(spec['roi']), gray0.shape[:2])
    parms = apply_aponeurosis_maxangle_overrides(
        parms,
        apo_maxangle=metadata.get('apo_fit_maxangle_deep_deg'),
    )
    rows.append({
        'dataset': name,
        'timtrack_fascicle_range_deg': tuple(np.asarray(parms['fas']['range'], dtype=float).reshape(-1)),
        'seed_grid_deg': (seed_cfg.angle_min_deg, seed_cfg.angle_max_deg),
        'apo_super_maxangle_deg': metadata.get('apo_fit_maxangle_super_deg'),
        'apo_deep_maxangle_deg': metadata.get('apo_fit_maxangle_deep_deg'),
        'selected_seed_alpha_deg': metadata.get('selected_seed_alpha_deg'),
        'kalman_mode': metadata.get('kalman_mode'),
    })

pd.DataFrame(rows)


,dataset,timtrack_fascicle_range_deg,seed_grid_deg,apo_super_maxangle_deg,apo_deep_maxangle_deg,selected_seed_alpha_deg,kalman_mode
0,test_23june_2,"(5.0, 60.0)","(14.0, 24.0)",20.0,20.0,20.2,adaptive-anisotropic
1,test23_june_3,"(5.0, 60.0)","(14.0, 24.0)",20.0,20.0,20.8,adaptive-anisotropic


## 3. Helper Functions


In [3]:
def read_gray_frame(video_path: Path, frame_idx: int) -> np.ndarray:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f'Could not read frame {frame_idx} from {video_path}')
    return cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame.copy()


def video_info(video_path: Path) -> dict[str, float | int]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return {'fps': fps, 'n_frames': n_frames, 'width': width, 'height': height}


def load_npz(path: Path) -> dict[str, np.ndarray]:
    with np.load(path) as data:
        return {key: data[key] for key in data.files}


def line_mid_y(lines: np.ndarray) -> np.ndarray:
    arr = np.asarray(lines, dtype=float)
    return (arr[:, 1] + arr[:, 3]) / 2.0


def line_angle_deg(lines: np.ndarray) -> np.ndarray:
    arr = np.asarray(lines, dtype=float)
    slope = (arr[:, 3] - arr[:, 1]) / np.maximum(arr[:, 2] - arr[:, 0], 1e-12)
    return -np.rad2deg(np.arctan2(slope, 1.0))


def segment_length_px(segments: np.ndarray) -> np.ndarray:
    arr = np.asarray(segments, dtype=float)
    return np.hypot(arr[:, 2] - arr[:, 0], arr[:, 3] - arr[:, 1])


def one_based_to_int_points(segment_1b: np.ndarray) -> tuple[tuple[int, int], tuple[int, int]] | None:
    seg = np.asarray(segment_1b, dtype=float).reshape(-1)
    if seg.size < 4 or not np.all(np.isfinite(seg[:4])):
        return None
    p1 = (int(round(seg[0] - 1.0)), int(round(seg[1] - 1.0)))
    p2 = (int(round(seg[2] - 1.0)), int(round(seg[3] - 1.0)))
    return p1, p2


def draw_line_1b(vis: np.ndarray, segment_1b: np.ndarray, color: tuple[int, int, int], thickness: int = 2) -> None:
    pts = one_based_to_int_points(segment_1b)
    if pts is None:
        return
    cv2.line(vis, pts[0], pts[1], color, int(thickness), cv2.LINE_AA)


def draw_text(vis: np.ndarray, lines: Sequence[str], x: int = 18, y: int = 28) -> None:
    for idx, text in enumerate(lines):
        yy = y + idx * 22
        cv2.putText(vis, text, (x, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(vis, text, (x, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)


def transform_endpoint_state(state_y: np.ndarray, affine: np.ndarray, width: int) -> np.ndarray:
    superficial, deep = lines_from_aponeurosis_state(state_y, width)
    points = np.asarray([
        [superficial[0], superficial[1]],
        [superficial[2], superficial[3]],
        [deep[0], deep[1]],
        [deep[2], deep[3]],
    ], dtype=np.float64)
    transformed = apply_affine_1b(points, affine)
    return np.asarray([transformed[0, 1], transformed[1, 1], transformed[2, 1], transformed[3, 1]], dtype=np.float64)


def frame_windows_mask(n: int, frames: Sequence[int]) -> np.ndarray:
    mask = np.zeros(n, dtype=bool)
    valid = [f for f in frames if 0 <= int(f) < n]
    mask[np.asarray(valid, dtype=int)] = True
    return mask


## 4. Baseline Strict Output

This loads the existing `run_strict_ultratimtrack_video.py` outputs and quantifies the reported windows before any new gating.


In [4]:
baselines: dict[str, dict[str, np.ndarray]] = {}
metadata_by_dataset = {}

summary_rows = []
window_rows = []
for name, spec in DATASETS.items():
    arrays = load_npz(spec['run_dir'] / f'{name}_strict_results.npz')
    metadata = json.loads((spec['run_dir'] / f'{name}_strict_metadata.json').read_text())
    baselines[name] = arrays
    metadata_by_dataset[name] = metadata
    n = len(arrays['frame'])
    fl = arrays.get('FL_mm', arrays['FL_px'])
    ang = arrays['ANG_deg']
    deep_mid = line_mid_y(arrays['deep_apo_lines'])
    deep_angle = line_angle_deg(arrays['deep_apo_lines'])
    problem = frame_windows_mask(n, spec['problem_frames'])
    outside = ~problem
    summary_rows.append({
        'dataset': name,
        'n_frames': n,
        'baseline_FL_range_all': float(np.nanmax(fl) - np.nanmin(fl)),
        'baseline_FL_range_problem': float(np.nanmax(fl[problem]) - np.nanmin(fl[problem])),
        'baseline_max_abs_FL_step_problem': float(np.nanmax(np.abs(np.diff(fl, prepend=fl[0]))[problem])),
        'baseline_max_abs_ANG_step_problem': float(np.nanmax(np.abs(np.diff(ang, prepend=ang[0]))[problem])),
        'baseline_max_abs_deep_mid_step_problem_px': float(np.nanmax(np.abs(np.diff(deep_mid, prepend=deep_mid[0]))[problem])),
        'baseline_max_abs_deep_angle_step_problem_deg': float(np.nanmax(np.abs(np.diff(deep_angle, prepend=deep_angle[0]))[problem])),
        'median_FL_outside_problem': float(np.nanmedian(fl[outside])),
        'median_ANG_outside_problem': float(np.nanmedian(ang[outside])),
    })
    for frame in spec['problem_frames']:
        if 0 <= frame < n:
            window_rows.append({
                'dataset': name,
                'frame': frame,
                'FL_mm_or_px': float(fl[frame]),
                'ANG_deg': float(ang[frame]),
                'PEN_deg': float(arrays['PEN_deg'][frame]),
                'TimTrackAlpha_deg': float(arrays['timtrack_alpha_deg'][frame]),
                'deep_mid_y_px': float(deep_mid[frame]),
                'deep_angle_deg': float(deep_angle[frame]),
                'deep_mid_step_px': float(np.diff(deep_mid, prepend=deep_mid[0])[frame]),
            })

summary_df = pd.DataFrame(summary_rows)
window_df = pd.DataFrame(window_rows)
summary_df


,dataset,n_frames,baseline_FL_range_all,baseline_FL_range_problem,baseline_max_abs_FL_step_problem,baseline_max_abs_ANG_step_problem,baseline_max_abs_deep_mid_step_problem_px,baseline_max_abs_deep_angle_step_problem_deg,median_FL_outside_problem,median_ANG_outside_problem
0,test_23june_2,197,37.752865,25.291518,13.710646,0.896759,6.356795,0.343315,69.530914,21.539401
1,test23_june_3,138,40.950964,13.114962,10.489112,0.309442,7.169964,1.168870,72.786453,21.657688


In [5]:
window_df


,dataset,frame,FL_mm_or_px,ANG_deg,PEN_deg,TimTrackAlpha_deg,deep_mid_y_px,deep_angle_deg,deep_mid_step_px
0,test_23june_2,102,82.718542,25.719744,5.746810,11.5,418.947485,19.972935,-0.201710
1,test_23june_2,103,81.768304,25.795757,5.805610,14.0,418.872710,19.990147,-0.074775
2,test_23june_2,104,82.349869,25.764728,5.748256,21.5,419.127863,20.016473,0.255153
3,test_23june_2,105,85.436052,25.653760,5.497044,21.5,420.882190,20.156716,1.754327
4,test_23june_2,106,90.448772,25.393049,5.256259,11.0,424.080667,20.136789,3.198477
5,test_23june_2,107,95.635300,25.488007,5.007903,18.0,430.437462,20.480104,6.356795
6,test_23june_2,108,81.924654,26.384766,6.090795,16.0,435.243960,20.293970,4.806498
7,test_23june_2,109,75.208220,26.887960,6.828968,17.0,438.290618,20.058992,3.046658
8,test_23june_2,110,70.343782,27.448840,7.434130,17.5,441.164072,20.014710,2.873454
9,test23_june_3,88,72.774293,25.431409,7.045841,13.0,410.141708,18.385568,0.947178


## 5. Recreate Raw TimTrack Measurements

The existing NPZ contains the filtered aponeurosis and final fascicle outputs, but to test measurement rejection we need the raw TimTrack measurement stream: aponeurosis measurement lines, alpha peaks, Hough candidate lines, and weights.

The first run computes these arrays; later runs load the cache.


In [6]:
def prepare_parms_for_dataset(name: str) -> dict:
    spec = DATASETS[name]
    frame0 = read_gray_frame(spec['video'], 0)
    parms = copy.deepcopy(mat_root['parms'])
    parms = update_parms_from_rois(parms, roi.load_rois(spec['roi']), frame0.shape[:2])
    metadata = metadata_by_dataset[name]
    parms = apply_aponeurosis_maxangle_overrides(
        parms,
        apo_maxangle=metadata.get('apo_fit_maxangle_deep_deg'),
    )
    return parms


def compact_entries_to_arrays(entries: list[Mapping], width: int) -> dict[str, np.ndarray]:
    super_lines, deep_lines = geofeature_measurement_lines(entries, width)
    alpha = geofeature_alpha(entries)
    n = len(entries)
    peak_alphas = np.full((n, 10), np.nan, dtype=float)
    peak_weights = np.full((n, 10), np.nan, dtype=float)
    cand_x = np.full((n, 10, 2), np.nan, dtype=float)
    cand_y = np.full((n, 10, 2), np.nan, dtype=float)
    for idx, entry in enumerate(entries):
        a = np.asarray(entry.get('alphas', []), dtype=float).reshape(-1)
        w = np.asarray(entry.get('weights', entry.get('ws', [])), dtype=float).reshape(-1)
        x = np.asarray(entry.get('x', []), dtype=float)
        y = np.asarray(entry.get('y', []), dtype=float)
        m = min(10, len(a))
        if m:
            peak_alphas[idx, :m] = a[:m]
        m = min(10, len(w))
        if m:
            peak_weights[idx, :m] = w[:m]
        if x.ndim == 2:
            m = min(10, x.shape[0], x.shape[1] if x.shape[1] < 2 else x.shape[0])
            cand_x[idx, :min(10, x.shape[0]), :2] = x[:10, :2]
        if y.ndim == 2:
            cand_y[idx, :min(10, y.shape[0]), :2] = y[:10, :2]
    return {
        'alpha': alpha,
        'peak_alphas': peak_alphas,
        'peak_weights': peak_weights,
        'candidate_x': cand_x,
        'candidate_y': cand_y,
        'super_measurement_lines': super_lines,
        'deep_measurement_lines': deep_lines,
        'super_measurement_mid_y': line_mid_y(super_lines),
        'deep_measurement_mid_y': line_mid_y(deep_lines),
        'super_measurement_angle_deg': line_angle_deg(super_lines),
        'deep_measurement_angle_deg': line_angle_deg(deep_lines),
    }


def arrays_to_geofeatures(meas: Mapping[str, np.ndarray]) -> list[dict[str, np.ndarray]]:
    entries = []
    n = len(meas['alpha'])
    for idx in range(n):
        entries.append({
            'frame': idx,
            'alpha': float(meas['alpha'][idx]),
            'alphas': meas['peak_alphas'][idx],
            'weights': meas['peak_weights'][idx],
            'ws': meas['peak_weights'][idx],
            'x': meas['candidate_x'][idx],
            'y': meas['candidate_y'][idx],
            'super_pos': np.asarray([meas['super_measurement_lines'][idx, 1], meas['super_measurement_lines'][idx, 3]], dtype=float),
            'deep_pos': np.asarray([meas['deep_measurement_lines'][idx, 1], meas['deep_measurement_lines'][idx, 3]], dtype=float),
        })
    return entries


def run_or_load_timtrack_measurements(name: str) -> dict[str, np.ndarray]:
    spec = DATASETS[name]
    out_dir = OUT_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    cache_path = out_dir / f'{name}_raw_timtrack_measurements.npz'
    if cache_path.exists():
        print('loading raw measurement cache:', cache_path)
        return load_npz(cache_path)

    info = video_info(spec['video'])
    parms = prepare_parms_for_dataset(name)
    print('computing raw TimTrack measurements:', name)
    t0 = time.time()
    entries = run_timtrack_geofeatures_from_video(
        str(spec['video']),
        parms,
        limit=int(info['n_frames']),
        keep_debug=False,
        emask_mode='matlab',
        progress_every=50,
    )
    meas = compact_entries_to_arrays(entries, int(info['width']))
    np.savez_compressed(cache_path, **meas)
    print(f'saved {cache_path} in {time.time() - t0:.1f}s')
    return meas

measurements = {name: run_or_load_timtrack_measurements(name) for name in DATASETS}

raw_rows = []
for name, meas in measurements.items():
    for side in ['super', 'deep']:
        mid = meas[f'{side}_measurement_mid_y']
        ang = meas[f'{side}_measurement_angle_deg']
        raw_rows.append({
            'dataset': name,
            'side': side,
            'mid_y_min': float(np.nanmin(mid)),
            'mid_y_max': float(np.nanmax(mid)),
            'max_abs_mid_step_px': float(np.nanmax(np.abs(np.diff(mid, prepend=mid[0])))),
            'angle_min_deg': float(np.nanmin(ang)),
            'angle_max_deg': float(np.nanmax(ang)),
            'max_abs_angle_step_deg': float(np.nanmax(np.abs(np.diff(ang, prepend=ang[0])))),
        })

pd.DataFrame(raw_rows)


loading raw measurement cache: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/test_23june_2_raw_timtrack_measurements.npz
loading raw measurement cache: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test23_june_3/test23_june_3_raw_timtrack_measurements.npz


,dataset,side,mid_y_min,mid_y_max,max_abs_mid_step_px,angle_min_deg,angle_max_deg,max_abs_angle_step_deg
0,test_23june_2,super,104.3,112.0,3.0,-0.345185,2.879940,1.687688
1,test_23june_2,deep,417.7,479.7,12.3,0.556083,20.000000,4.757289
2,test23_june_3,super,102.6,109.5,4.0,-0.135449,2.920934,1.654724
3,test23_june_3,deep,406.3,463.2,17.5,1.689433,19.936973,7.048751


## 6. Candidate Fix A: Aponeurosis Innovation and Jump Gate

This variant re-runs the aponeurosis state filter locally with extra measurement diagnostics:

- separate endpoint measurement variance scales for superficial-left, superficial-right, deep-left, deep-right;
- hard rejection when measurement innovation or frame-to-frame measurement jump is too large;
- stronger deep-aponeurosis penalty because FL is reconstructed from the deep intersection;
- rejection logs for every frame.

Rejected measurements are not smoothed post hoc; the filter uses the KLT-predicted prior for that component for that frame.


In [7]:
@dataclass(frozen=True)
class ApoGateConfig:
    endpoint_innovation_px: float = 32.0
    endpoint_jump_px: float = 32.0
    mid_innovation_px: float = 10.0
    deep_mid_jump_px: float = 6.0
    super_mid_jump_px: float = 12.0
    deep_angle_jump_deg: float = 2.5
    super_angle_jump_deg: float = 3.5
    near_maxangle_fraction: float = 0.90
    giant_mid_jump_px: float = 16.0
    giant_mid_innovation_px: float = 24.0
    soft_fraction: float = 0.65
    soft_r_scale: float = 12.0
    hard_r_scale: float = 1.0e6


def one_line_mid_angle_from_state(state: np.ndarray, width: int, side: str) -> tuple[float, float]:
    super_line, deep_line = lines_from_aponeurosis_state(state, width)
    line = super_line if side == 'super' else deep_line
    mid = float((line[1] + line[3]) / 2.0)
    angle = float(line_angle_deg(line.reshape(1, 4))[0])
    return mid, angle


def run_gated_aponeurosis_state_video(
    name: str,
    geofeatures: list[Mapping],
    super_meas: np.ndarray,
    deep_meas: np.ndarray,
    *,
    config: ApoGateConfig,
) -> dict[str, np.ndarray]:
    spec = DATASETS[name]
    info = video_info(spec['video'])
    frame0 = read_gray_frame(spec['video'], 0)
    shape = frame0.shape[:2]
    width = int(info['width'])
    parms = prepare_parms_for_dataset(name)
    super_cut = np.asarray(parms['apo']['super']['cut'], dtype=float).reshape(-1)
    deep_cut = np.asarray(parms['apo']['deep']['cut'], dtype=float).reshape(-1)
    metadata = metadata_by_dataset[name]
    super_maxangle = float(metadata.get('apo_fit_maxangle_super_deg', np.nan))
    deep_maxangle = float(metadata.get('apo_fit_maxangle_deep_deg', np.nan))

    block_size = np.asarray(mat_root.get('BlockSize', [81, 81]), dtype=int).reshape(-1)
    win_size = (int(block_size[-1]), int(block_size[0])) if block_size.size >= 2 else (81, 81)
    klt_config = UltraTrackKLTConfig(lk_win_size=win_size)
    r_values = np.asarray(mat_root.get('R', [3.05529211, 100, 100, 100, 100]), dtype=float).reshape(-1)
    measurement_variance = r_values[1:5] * 0.01 if r_values.size >= 5 else np.ones(4, dtype=float)
    q_parameter = float(mat_root.get('Q', 0.01))

    n = min(len(geofeatures), len(super_meas), len(deep_meas), int(info['n_frames']))
    cap = cv2.VideoCapture(str(spec['video']))
    if not cap.isOpened():
        raise FileNotFoundError(spec['video'])
    ok, prev_frame = cap.read()
    if not ok:
        raise RuntimeError(f'Could not read first frame from {spec["video"]}')
    prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY) if prev_frame.ndim == 3 else prev_frame.copy()

    states_plus = np.full((n, 4), np.nan, dtype=float)
    states_minus = np.full((n, 4), np.nan, dtype=float)
    p_plus = np.full((n, 4), np.nan, dtype=float)
    p_minus = np.full((n, 4), np.nan, dtype=float)
    gains = np.full((n, 4), np.nan, dtype=float)
    r_scales = np.ones((n, 4), dtype=float)
    rejected = np.zeros((n, 4), dtype=bool)
    soft_downweighted = np.zeros((n, 4), dtype=bool)
    line_rejected = np.zeros((n, 2), dtype=bool)
    reasons = np.empty((n, 2), dtype=object)
    reasons[:] = ''
    super_lines = np.full((n, 4), np.nan, dtype=float)
    deep_lines = np.full((n, 4), np.nan, dtype=float)
    prior_super_lines = np.full((n, 4), np.nan, dtype=float)
    prior_deep_lines = np.full((n, 4), np.nan, dtype=float)
    points_count = np.zeros(n, dtype=np.int32)
    tracked_count = np.zeros(n, dtype=np.int32)
    inlier_count = np.zeros(n, dtype=np.int32)
    affine_ok = np.zeros(n, dtype=bool)

    states_plus[0] = aponeurosis_state_from_lines(super_meas[0], deep_meas[0])
    states_minus[0] = states_plus[0]
    p_plus[0] = measurement_variance
    p_minus[0] = measurement_variance
    super_lines[0], deep_lines[0] = lines_from_aponeurosis_state(states_plus[0], width)
    prior_super_lines[0], prior_deep_lines[0] = super_lines[0], deep_lines[0]
    prev_accepted_measurement = states_plus[0].copy()

    for frame in range(1, n):
        ok, current_frame = cap.read()
        if not ok:
            n = frame
            break
        gray = cv2.cvtColor(current_frame, cv2.COLOR_BGR2GRAY) if current_frame.ndim == 3 else current_frame.copy()
        masks = tracking_masks_from_geofeature(
            geofeatures[frame - 1],
            shape=shape,
            super_cut=super_cut,
            deep_cut=deep_cut,
        )
        points = detect_min_eigen_like(
            prev_gray,
            masks['apo_mask'],
            max_corners=klt_config.max_aponeurosis_corners,
            config=klt_config,
        )
        points = filter_points_by_mask(points, masks['apo_mask'])
        old_points, new_points = track_points(prev_gray, gray, points, config=klt_config)
        affine, inliers = estimate_affine_matlab_coords(old_points, new_points, config=klt_config)
        points_count[frame] = len(points)
        tracked_count[frame] = len(new_points)
        inlier_count[frame] = int(inliers)
        if affine is not None:
            prior_state = transform_endpoint_state(states_plus[frame - 1], affine, width)
            affine_ok[frame] = True
        else:
            prior_state = states_plus[frame - 1].copy()

        states_minus[frame] = prior_state
        prior_super_lines[frame], prior_deep_lines[frame] = lines_from_aponeurosis_state(prior_state, width)
        measurement_state = aponeurosis_state_from_lines(super_meas[frame], deep_meas[frame])
        innovation = measurement_state - prior_state
        measurement_jump = measurement_state - prev_accepted_measurement
        candidate_measurement = measurement_state.copy()
        frame_r_scale = np.ones(4, dtype=float)
        frame_reject = np.zeros(4, dtype=bool)
        frame_soft = np.zeros(4, dtype=bool)

        for side_name, side_idx, endpoint_slice in [('super', 0, slice(0, 2)), ('deep', 1, slice(2, 4))]:
            idxs = np.arange(4)[endpoint_slice]
            meas_line = super_meas[frame] if side_name == 'super' else deep_meas[frame]
            prev_line = lines_from_aponeurosis_state(prev_accepted_measurement, width)[0 if side_name == 'super' else 1]
            prior_mid, prior_angle = one_line_mid_angle_from_state(prior_state, width, side_name)
            meas_mid = float((meas_line[1] + meas_line[3]) / 2.0)
            prev_mid = float((prev_line[1] + prev_line[3]) / 2.0)
            meas_angle = float(line_angle_deg(meas_line.reshape(1, 4))[0])
            prev_angle = float(line_angle_deg(prev_line.reshape(1, 4))[0])
            mid_innovation = abs(meas_mid - prior_mid)
            mid_jump = abs(meas_mid - prev_mid)
            angle_jump = abs(meas_angle - prev_angle)
            mid_jump_limit = config.deep_mid_jump_px if side_name == 'deep' else config.super_mid_jump_px
            angle_jump_limit = config.deep_angle_jump_deg if side_name == 'deep' else config.super_angle_jump_deg
            reason_parts = []

            side_maxangle = deep_maxangle if side_name == 'deep' else super_maxangle
            near_maxangle = (
                np.isfinite(side_maxangle)
                and side_maxangle > 0
                and abs(meas_angle) >= config.near_maxangle_fraction * side_maxangle
            )
            huge_mid = mid_jump > config.giant_mid_jump_px or mid_innovation > config.giant_mid_innovation_px
            suspicious_mid = mid_jump > mid_jump_limit or mid_innovation > config.mid_innovation_px
            suspicious_angle = angle_jump > angle_jump_limit

            hard_line = False
            if side_name == 'deep':
                hard_line = bool(huge_mid or (near_maxangle and (suspicious_mid or suspicious_angle)))
            else:
                hard_line = bool(huge_mid and (suspicious_mid or suspicious_angle))

            if suspicious_mid:
                reason_parts.append(f'mid jump/innovation {mid_jump:.1f}/{mid_innovation:.1f}px')
            if suspicious_angle:
                reason_parts.append(f'angle jump {angle_jump:.1f}deg')
            if near_maxangle:
                reason_parts.append(f'near maxangle {meas_angle:.1f}deg')
            if huge_mid:
                reason_parts.append('giant mid displacement')

            if hard_line:
                candidate_measurement[idxs] = prior_state[idxs]
                frame_r_scale[idxs] = config.hard_r_scale
                frame_reject[idxs] = True
                line_rejected[frame, side_idx] = True
                reasons[frame, side_idx] = '; '.join(reason_parts)
            else:
                soft_line = bool(suspicious_mid or suspicious_angle)
                for idx in idxs:
                    endpoint_bad = abs(innovation[idx]) > config.endpoint_innovation_px or abs(measurement_jump[idx]) > config.endpoint_jump_px
                    endpoint_soft = (
                        soft_line
                        or abs(innovation[idx]) > config.endpoint_innovation_px * config.soft_fraction
                        or abs(measurement_jump[idx]) > config.endpoint_jump_px * config.soft_fraction
                    )
                    if endpoint_bad and near_maxangle:
                        candidate_measurement[idx] = prior_state[idx]
                        frame_r_scale[idx] = config.hard_r_scale
                        frame_reject[idx] = True
                    elif endpoint_soft:
                        frame_r_scale[idx] = max(frame_r_scale[idx], config.soft_r_scale)
                        frame_soft[idx] = True

        delta = np.abs(prior_state - states_plus[frame - 1])
        for idx in range(4):
            states_plus[frame, idx], p_plus[frame, idx], p_minus[frame, idx], gains[frame, idx] = matlab_scalar_kalman_update(
                prior_state[idx],
                p_plus[frame - 1, idx],
                q_parameter * float(delta[idx]) ** 2,
                candidate_measurement[idx],
                measurement_variance[idx] * frame_r_scale[idx],
            )

        posterior_deep_line = lines_from_aponeurosis_state(states_plus[frame], width)[1]
        previous_deep_line = lines_from_aponeurosis_state(states_plus[frame - 1], width)[1]
        posterior_mid = float((posterior_deep_line[1] + posterior_deep_line[3]) / 2.0)
        previous_mid = float((previous_deep_line[1] + previous_deep_line[3]) / 2.0)
        posterior_angle = float(line_angle_deg(posterior_deep_line.reshape(1, 4))[0])
        posterior_near_max = (
            np.isfinite(deep_maxangle)
            and deep_maxangle > 0
            and abs(posterior_angle) >= config.near_maxangle_fraction * deep_maxangle
        )
        posterior_jump = abs(posterior_mid - previous_mid)
        if posterior_near_max and posterior_jump > config.deep_mid_jump_px:
            states_plus[frame, 2:4] = prior_state[2:4]
            frame_r_scale[2:4] = config.hard_r_scale
            frame_reject[2:4] = True
            line_rejected[frame, 1] = True
            post_reason = f'posterior deep jump {posterior_jump:.1f}px near maxangle {posterior_angle:.1f}deg'
            reasons[frame, 1] = (str(reasons[frame, 1]) + '; ' + post_reason).strip('; ')

        r_scales[frame] = frame_r_scale
        rejected[frame] = frame_reject
        soft_downweighted[frame] = frame_soft
        accepted_for_jump = measurement_state.copy()
        accepted_for_jump[frame_reject] = prev_accepted_measurement[frame_reject]
        prev_accepted_measurement = accepted_for_jump
        super_lines[frame], deep_lines[frame] = lines_from_aponeurosis_state(states_plus[frame], width)
        prev_gray = gray

    cap.release()
    sl = slice(0, n)
    return {
        'super_lines': super_lines[sl],
        'deep_lines': deep_lines[sl],
        'prior_super_lines': prior_super_lines[sl],
        'prior_deep_lines': prior_deep_lines[sl],
        'states_plus': states_plus[sl],
        'states_minus': states_minus[sl],
        'p_plus': p_plus[sl],
        'p_minus': p_minus[sl],
        'gains': gains[sl],
        'r_scales': r_scales[sl],
        'rejected_endpoints': rejected[sl],
        'soft_downweighted_endpoints': soft_downweighted[sl],
        'line_rejected': line_rejected[sl],
        'reasons': reasons[sl],
        'points_count': points_count[sl],
        'tracked_count': tracked_count[sl],
        'inlier_count': inlier_count[sl],
        'affine_ok': affine_ok[sl],
    }


## 7. Candidate Fix B: Alpha Candidate Persistence and Configurable Steep Range

The Hough detector already searches 5 to 60 degrees. This experiment uses that broad range but avoids abrupt one-frame alpha changes by preferring the candidate closest to the previous accepted alpha, with a small bonus for stronger Hough weight.

This tests the idea, but it is intentionally kept as a separate variant because rejecting steep alpha changes can be harmful if the anatomy truly moves steeply.


In [8]:
@dataclass(frozen=True)
class AlphaPersistenceConfig:
    angle_min_deg: float = 5.0
    angle_max_deg: float = 60.0
    max_preferred_jump_deg: float = 8.0
    max_hard_jump_deg: float = 14.0
    hough_weight_bonus_deg: float = 2.0
    rejection_theta_r_scale: float = 30.0


def persistent_alpha_stream(meas: Mapping[str, np.ndarray], config: AlphaPersistenceConfig) -> dict[str, np.ndarray]:
    raw = np.asarray(meas['alpha'], dtype=float).copy()
    peaks = np.asarray(meas['peak_alphas'], dtype=float)
    weights = np.asarray(meas['peak_weights'], dtype=float)
    n = len(raw)
    selected = raw.copy()
    selected_peak_idx = np.full(n, -1, dtype=int)
    raw_rejected = np.zeros(n, dtype=bool)
    theta_r_scale = np.ones(n, dtype=float)
    reason = np.empty(n, dtype=object)
    reason[:] = ''

    if n == 0:
        return {
            'alpha': selected,
            'selected_peak_idx': selected_peak_idx,
            'raw_rejected': raw_rejected,
            'theta_r_scale': theta_r_scale,
            'reason': reason,
        }

    selected[0] = raw[0]
    for idx in range(1, n):
        prev = selected[idx - 1]
        row = peaks[idx]
        row_weights = weights[idx]
        valid = np.isfinite(row) & (row >= config.angle_min_deg) & (row <= config.angle_max_deg)
        if not np.any(valid):
            selected[idx] = raw[idx]
            continue
        cand = row[valid]
        cand_original_idx = np.flatnonzero(valid)
        w = row_weights[valid]
        if np.any(np.isfinite(w)) and np.nanmax(w) > 0:
            w_norm = np.nan_to_num(w / np.nanmax(w), nan=0.0)
        else:
            w_norm = np.zeros_like(cand)
        raw_jump = abs(raw[idx] - prev) if np.isfinite(raw[idx]) and np.isfinite(prev) else 0.0
        costs = np.abs(cand - prev) - config.hough_weight_bonus_deg * w_norm
        best_pos = int(np.nanargmin(costs))
        best_alpha = float(cand[best_pos])
        best_jump = abs(best_alpha - prev)

        if raw_jump <= config.max_preferred_jump_deg or not np.isfinite(raw_jump):
            selected[idx] = raw[idx]
        elif best_jump <= config.max_hard_jump_deg:
            selected[idx] = best_alpha
            selected_peak_idx[idx] = int(cand_original_idx[best_pos])
            raw_rejected[idx] = abs(best_alpha - raw[idx]) > 0.25
            theta_r_scale[idx] = config.rejection_theta_r_scale if raw_rejected[idx] else 1.0
            reason[idx] = f'raw jump {raw_jump:.1f}deg; chose candidate {best_alpha:.1f}deg near previous {prev:.1f}deg'
        else:
            selected[idx] = raw[idx]
            theta_r_scale[idx] = config.rejection_theta_r_scale
            reason[idx] = f'raw jump {raw_jump:.1f}deg; no persistent candidate within {config.max_hard_jump_deg:.1f}deg'

    return {
        'alpha': selected,
        'selected_peak_idx': selected_peak_idx,
        'raw_rejected': raw_rejected,
        'theta_r_scale': theta_r_scale,
        'reason': reason,
    }


## 8. Run Variants

Both variants keep the saved KLT prior from the existing strict run. The only changes under test are measurement handling:

- `apo_gate`: gated aponeurosis lines, original TimTrack alpha.
- `apo_gate_alpha_persist`: gated aponeurosis lines, persistent alpha candidate stream, extra theta down-weighting when raw alpha is rejected.


In [9]:
apo_gate_cfg = ApoGateConfig()
alpha_cfg = AlphaPersistenceConfig(angle_min_deg=5.0, angle_max_deg=60.0)
variants: dict[str, dict[str, dict[str, np.ndarray]]] = {name: {} for name in DATASETS}
variant_logs: dict[str, dict[str, np.ndarray]] = {}

r_values = np.asarray(mat_root.get('R', [3.05529211]), dtype=float).reshape(-1)
kalman_cfg = MatlabTwoStateKalmanConfig(
    q_parameter=float(mat_root.get('Q', 0.01)),
    x_measurement_variance=float(mat_root.get('X', 100.0)),
    alpha_measurement_variance=float(r_values[0]),
    n_start_frames=int(mat_root.get('NS', 1)),
    run_smoother=True,
    use_adaptive_R=True,
)

for name, spec in DATASETS.items():
    print('\n' + '=' * 80)
    print(name)
    baseline = baselines[name]
    meas = measurements[name]
    geofeatures = arrays_to_geofeatures(meas)
    super_meas = meas['super_measurement_lines']
    deep_meas = meas['deep_measurement_lines']
    t0 = time.time()
    gated_apo = run_gated_aponeurosis_state_video(
        name,
        geofeatures,
        super_meas,
        deep_meas,
        config=apo_gate_cfg,
    )
    print(f'gated aponeurosis in {time.time() - t0:.1f}s')
    alpha_persist = persistent_alpha_stream(meas, alpha_cfg)
    variant_logs[name] = {'apo': gated_apo, 'alpha_persist': alpha_persist}

    n = len(gated_apo['deep_lines'])
    mm_per_px = float(np.asarray(baseline['mm_per_pixel']).reshape(-1)[0])
    base_r_scale = np.asarray(baseline.get('r_scale', np.ones(n)), dtype=float)[:n]
    base_theta = np.asarray(baseline.get('r_scale_theta', np.ones(n)), dtype=float)[:n]
    base_length = np.asarray(baseline.get('r_scale_length', np.ones(n)), dtype=float)[:n]
    length_extra = np.maximum(1.0, np.nanmax(gated_apo['r_scales'], axis=1))
    length_extra = np.clip(length_extra, 1.0, 50.0)

    apo_gate = run_matlab_2state_kalman(
        baseline['klt_prior_segments'][:n],
        baseline['timtrack_alpha_deg'][:n],
        gated_apo['super_lines'],
        gated_apo['deep_lines'],
        config=kalman_cfg,
        mm_per_pixel=mm_per_px,
        measurement_r_scale=base_r_scale,
        measurement_r_scale_theta=base_theta,
        measurement_r_scale_length=np.maximum(base_length, length_extra),
    )
    variants[name]['apo_gate'] = apo_gate

    theta_extra = np.maximum(base_theta, np.asarray(alpha_persist['theta_r_scale'], dtype=float)[:n])
    apo_alpha = run_matlab_2state_kalman(
        baseline['klt_prior_segments'][:n],
        alpha_persist['alpha'][:n],
        gated_apo['super_lines'],
        gated_apo['deep_lines'],
        config=kalman_cfg,
        mm_per_pixel=mm_per_px,
        measurement_r_scale=base_r_scale,
        measurement_r_scale_theta=theta_extra,
        measurement_r_scale_length=np.maximum(base_length, length_extra),
    )
    variants[name]['apo_gate_alpha_persist'] = apo_alpha

    out_dir = OUT_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        out_dir / f'{name}_gated_aponeurosis_log.npz',
        **{k: v for k, v in gated_apo.items() if k != 'reasons'},
        reasons=gated_apo['reasons'].astype(str),
        alpha_persistent=alpha_persist['alpha'],
        alpha_raw_rejected=alpha_persist['raw_rejected'],
        alpha_reason=alpha_persist['reason'].astype(str),
    )
    print('deep line rejects:', int(gated_apo['line_rejected'][:, 1].sum()))
    print('alpha raw rejects:', int(alpha_persist['raw_rejected'].sum()))



test_23june_2


gated aponeurosis in 6.3s
deep line rejects: 54
alpha raw rejects: 29

test23_june_3


gated aponeurosis in 3.0s
deep line rejects: 49
alpha raw rejects: 21


## 9. Before/After Metrics

The preservation columns compare variants to the current baseline outside the annotated problem windows. Small outside-window deltas are desirable; large deltas mean the variant may be suppressing true motion.


In [10]:
def summarize_variant(name: str, variant_name: str, result: Mapping[str, np.ndarray]) -> dict[str, float | int | str]:
    baseline = baselines[name]
    n = min(len(result['ANG_deg']), len(baseline['ANG_deg']))
    problem = frame_windows_mask(n, DATASETS[name]['problem_frames'])
    outside = ~problem
    base_fl = baseline.get('FL_mm', baseline['FL_px'])[:n]
    var_fl = result.get('FL_mm', result['FL_px'])[:n]
    base_ang = baseline['ANG_deg'][:n]
    var_ang = result['ANG_deg'][:n]
    base_deep_mid = line_mid_y(baseline['deep_apo_lines'][:n])
    gated_deep_mid = line_mid_y(variant_logs[name]['apo']['deep_lines'][:n])
    apo_log = variant_logs[name]['apo']
    alpha_log = variant_logs[name]['alpha_persist']
    return {
        'dataset': name,
        'variant': variant_name,
        'problem_FL_range': float(np.nanmax(var_fl[problem]) - np.nanmin(var_fl[problem])),
        'baseline_problem_FL_range': float(np.nanmax(base_fl[problem]) - np.nanmin(base_fl[problem])),
        'problem_max_abs_FL_step': float(np.nanmax(np.abs(np.diff(var_fl, prepend=var_fl[0]))[problem])),
        'baseline_problem_max_abs_FL_step': float(np.nanmax(np.abs(np.diff(base_fl, prepend=base_fl[0]))[problem])),
        'problem_max_abs_ANG_step': float(np.nanmax(np.abs(np.diff(var_ang, prepend=var_ang[0]))[problem])),
        'baseline_problem_max_abs_ANG_step': float(np.nanmax(np.abs(np.diff(base_ang, prepend=base_ang[0]))[problem])),
        'problem_max_abs_deep_mid_step_px': float(np.nanmax(np.abs(np.diff(gated_deep_mid, prepend=gated_deep_mid[0]))[problem])),
        'baseline_problem_max_abs_deep_mid_step_px': float(np.nanmax(np.abs(np.diff(base_deep_mid, prepend=base_deep_mid[0]))[problem])),
        'outside_median_abs_FL_delta_vs_baseline': float(np.nanmedian(np.abs(var_fl[outside] - base_fl[outside]))),
        'outside_median_abs_ANG_delta_vs_baseline': float(np.nanmedian(np.abs(var_ang[outside] - base_ang[outside]))),
        'outside_p95_abs_FL_delta_vs_baseline': float(np.nanpercentile(np.abs(var_fl[outside] - base_fl[outside]), 95)),
        'outside_p95_abs_ANG_delta_vs_baseline': float(np.nanpercentile(np.abs(var_ang[outside] - base_ang[outside]), 95)),
        'deep_line_rejects_total': int(apo_log['line_rejected'][:, 1].sum()),
        'super_line_rejects_total': int(apo_log['line_rejected'][:, 0].sum()),
        'endpoint_rejects_total': int(apo_log['rejected_endpoints'].sum()),
        'alpha_raw_rejects_total': int(alpha_log['raw_rejected'].sum() if variant_name.endswith('alpha_persist') else 0),
    }

metric_rows = []
for name in DATASETS:
    for variant_name, result in variants[name].items():
        metric_rows.append(summarize_variant(name, variant_name, result))
metrics_df = pd.DataFrame(metric_rows)
metrics_path = OUT_ROOT / 'variant_metrics.csv'
metrics_df.to_csv(metrics_path, index=False)
metrics_df


,dataset,variant,problem_FL_range,baseline_problem_FL_range,problem_max_abs_FL_step,baseline_problem_max_abs_FL_step,problem_max_abs_ANG_step,baseline_problem_max_abs_ANG_step,problem_max_abs_deep_mid_step_px,baseline_problem_max_abs_deep_mid_step_px,outside_median_abs_FL_delta_vs_baseline,outside_median_abs_ANG_delta_vs_baseline,outside_p95_abs_FL_delta_vs_baseline,outside_p95_abs_ANG_delta_vs_baseline,deep_line_rejects_total,super_line_rejects_total,endpoint_rejects_total,alpha_raw_rejects_total
0,test_23june_2,apo_gate,16.342020,25.291518,8.011397,13.710646,0.896759,0.896759,2.006363,6.356795,0.125925,0.000000,11.668816,2.842171e-14,54,0,108,0
1,test_23june_2,apo_gate_alpha_persist,16.165613,25.291518,7.826497,13.710646,0.831223,0.896759,2.006363,6.356795,2.192331,0.191638,14.673788,5.502746e-01,54,0,108,29
2,test23_june_3,apo_gate,3.465910,13.114962,2.841916,10.489112,0.309442,0.309442,1.667862,7.169964,0.361681,0.000000,1870.623987,2.842171e-14,49,0,98,0
3,test23_june_3,apo_gate_alpha_persist,3.134528,13.114962,2.460998,10.489112,0.314746,0.309442,1.667862,7.169964,3.902877,0.527287,488.171247,6.320765e-01,49,0,98,21


In [11]:
def rejection_table(name: str) -> pd.DataFrame:
    apo = variant_logs[name]['apo']
    alpha = variant_logs[name]['alpha_persist']
    rows = []
    n = len(apo['line_rejected'])
    for idx in range(n):
        if apo['line_rejected'][idx].any() or apo['rejected_endpoints'][idx].any() or alpha['raw_rejected'][idx]:
            rows.append({
                'dataset': name,
                'frame': idx,
                'super_line_rejected': bool(apo['line_rejected'][idx, 0]),
                'deep_line_rejected': bool(apo['line_rejected'][idx, 1]),
                'endpoint_reject_count': int(apo['rejected_endpoints'][idx].sum()),
                'super_reason': str(apo['reasons'][idx, 0]),
                'deep_reason': str(apo['reasons'][idx, 1]),
                'raw_alpha': float(measurements[name]['alpha'][idx]),
                'persistent_alpha': float(alpha['alpha'][idx]),
                'alpha_rejected': bool(alpha['raw_rejected'][idx]),
                'alpha_reason': str(alpha['reason'][idx]),
            })
    return pd.DataFrame(rows)

reject_tables = []
for name in DATASETS:
    table = rejection_table(name)
    table.to_csv(OUT_ROOT / name / f'{name}_rejected_measurements.csv', index=False)
    reject_tables.append(table)

rejections_df = pd.concat(reject_tables, ignore_index=True)
rejections_df.head(40)


,dataset,frame,super_line_rejected,deep_line_rejected,endpoint_reject_count,super_reason,deep_reason,raw_alpha,persistent_alpha,alpha_rejected,alpha_reason
0,test_23june_2,35,False,False,0,,,14.5,27.5,True,raw jump 12.5deg; chose candidate 27.5deg near...
1,test_23june_2,36,False,False,0,,,14.5,27.5,True,raw jump 13.0deg; chose candidate 27.5deg near...
2,test_23june_2,37,False,False,0,,,14.5,27.5,True,raw jump 13.0deg; chose candidate 27.5deg near...
3,test_23june_2,38,False,False,0,,,10.0,14.5,True,raw jump 17.5deg; chose candidate 14.5deg near...
4,test_23june_2,59,False,False,0,,,30.5,14.5,True,raw jump 10.0deg; chose candidate 14.5deg near...
5,test_23june_2,60,False,False,0,,,30.5,14.5,True,raw jump 16.0deg; chose candidate 14.5deg near...
6,test_23june_2,66,False,False,0,,,8.0,16.5,True,raw jump 8.5deg; chose candidate 16.5deg near ...
7,test_23june_2,72,False,False,0,,,32.0,16.0,True,raw jump 15.5deg; chose candidate 16.0deg near...
8,test_23june_2,73,False,False,0,,,34.0,16.0,True,raw jump 18.0deg; chose candidate 16.0deg near...
9,test_23june_2,74,False,False,0,,,34.5,16.5,True,raw jump 18.5deg; chose candidate 16.5deg near...


## 10. Plots: FL, ANG, and Deep Aponeurosis Trajectory


In [12]:
def _local_ylim(values_list: list[np.ndarray], frame_mask: np.ndarray, *, pad_fraction: float = 0.12) -> tuple[float, float] | None:
    values = []
    for arr in values_list:
        a = np.asarray(arr, dtype=float)
        if len(a) == len(frame_mask):
            a = a[frame_mask]
        finite = a[np.isfinite(a)]
        if finite.size:
            values.append(finite)
    if not values:
        return None
    merged = np.concatenate(values)
    lo = float(np.nanmin(merged))
    hi = float(np.nanmax(merged))
    if not np.isfinite(lo) or not np.isfinite(hi):
        return None
    if abs(hi - lo) < 1e-9:
        pad = max(abs(hi) * 0.05, 1.0)
    else:
        pad = (hi - lo) * pad_fraction
    return lo - pad, hi + pad


def _plot_series_with_outlier_markers(ax, x, y, *, ylim: tuple[float, float], label: str, color: str, linewidth: float = 1.2):
    y = np.asarray(y, dtype=float)
    lo, hi = ylim
    inside = np.isfinite(y) & (y >= lo) & (y <= hi)
    y_plot = y.copy()
    y_plot[~inside] = np.nan
    ax.plot(x, y_plot, color=color, linewidth=linewidth, label=label)
    high = np.isfinite(y) & (y > hi)
    low = np.isfinite(y) & (y < lo)
    if np.any(high):
        ax.scatter(np.asarray(x)[high], np.full(int(high.sum()), hi), marker='^', s=18, color=color, zorder=6, label=f'{label} out of view ({int(high.sum())})')
    if np.any(low):
        ax.scatter(np.asarray(x)[low], np.full(int(low.sum()), lo), marker='v', s=18, color=color, zorder=6)


def plot_variant_comparison(name: str) -> list[Path]:
    baseline = baselines[name]
    apo = variant_logs[name]['apo']
    frames = baseline['frame'][:len(apo['deep_lines'])]
    base_fl = baseline.get('FL_mm', baseline['FL_px'])[:len(frames)]
    fl_label = 'FL (mm)' if 'FL_mm' in baseline else 'FL (px)'
    base_ang = baseline['ANG_deg'][:len(frames)]
    raw_deep_mid = measurements[name]['deep_measurement_mid_y'][:len(frames)]
    base_deep_mid = line_mid_y(baseline['deep_apo_lines'][:len(frames)])
    gated_deep_mid = line_mid_y(apo['deep_lines'])
    base_deep_ang = line_angle_deg(baseline['deep_apo_lines'][:len(frames)])
    gated_deep_ang = line_angle_deg(apo['deep_lines'])
    raw_deep_ang = measurements[name]['deep_measurement_angle_deg'][:len(frames)]

    problem = DATASETS[name]['problem_frames']
    zoom_lo = max(0, min(problem) - 4)
    zoom_hi = min(int(frames[-1]), max(problem) + 5)
    zoom_mask = (frames >= zoom_lo) & (frames <= zoom_hi)

    variant_colors = {'apo_gate': 'tab:blue', 'apo_gate_alpha_persist': 'tab:orange'}
    variant_fl = {variant_name: result.get('FL_mm', result['FL_px'])[:len(frames)] for variant_name, result in variants[name].items()}
    variant_ang = {variant_name: result['ANG_deg'][:len(frames)] for variant_name, result in variants[name].items()}

    fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=True)
    axes[0].plot(frames, base_fl, color='tab:red', linewidth=1.6, label='baseline strict')
    for variant_name, values in variant_fl.items():
        axes[0].plot(frames, values, linewidth=1.3, color=variant_colors.get(variant_name), label=variant_name)
    axes[0].set_ylabel(fl_label)
    ylim = _local_ylim([base_fl, *variant_fl.values()], zoom_mask)
    if ylim:
        axes[0].set_ylim(*ylim)

    axes[1].plot(frames, base_ang, color='tab:red', linewidth=1.5, label='baseline strict')
    axes[1].plot(frames, baseline['timtrack_alpha_deg'][:len(frames)], color='0.7', linewidth=0.9, label='raw TimTrack alpha')
    for variant_name, values in variant_ang.items():
        axes[1].plot(frames, values, linewidth=1.2, color=variant_colors.get(variant_name), label=variant_name)
    axes[1].set_ylabel('ANG (deg)')
    ylim = _local_ylim([base_ang, baseline['timtrack_alpha_deg'][:len(frames)], *variant_ang.values()], zoom_mask)
    if ylim:
        axes[1].set_ylim(*ylim)

    axes[2].plot(frames, raw_deep_mid, color='0.55', linewidth=0.9, label='raw deep measurement')
    axes[2].plot(frames, base_deep_mid, color='tab:red', linewidth=1.3, label='baseline filtered deep')
    axes[2].plot(frames, gated_deep_mid, color='tab:blue', linewidth=1.3, label='gated deep')
    rejected_frames = np.flatnonzero(apo['line_rejected'][:, 1])
    rejected_frames = rejected_frames[(rejected_frames >= zoom_lo) & (rejected_frames <= zoom_hi)]
    axes[2].scatter(rejected_frames, gated_deep_mid[rejected_frames], color='black', s=24, zorder=5, label='deep rejects')
    axes[2].set_ylabel('deep mid-y (px)')
    ylim = _local_ylim([raw_deep_mid, base_deep_mid, gated_deep_mid], zoom_mask)
    if ylim:
        axes[2].set_ylim(*ylim)

    axes[3].plot(frames, base_deep_ang, color='tab:red', linewidth=1.3, label='baseline deep angle')
    axes[3].plot(frames, gated_deep_ang, color='tab:blue', linewidth=1.3, label='gated deep angle')
    axes[3].plot(frames, raw_deep_ang, color='0.6', linewidth=0.9, label='raw deep angle')
    axes[3].set_ylabel('deep angle (deg)')
    axes[3].set_xlabel('frame')
    ylim = _local_ylim([base_deep_ang, gated_deep_ang, raw_deep_ang], zoom_mask)
    if ylim:
        axes[3].set_ylim(*ylim)

    for ax in axes:
        ax.axvspan(min(problem) - 0.5, max(problem) + 0.5, color='gold', alpha=0.16, linewidth=0)
        ax.set_xlim(zoom_lo, zoom_hi)
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best', fontsize=8)
    fig.suptitle(f'{name}: problem-window before/after anatomical gates')
    fig.tight_layout()
    out_zoom = OUT_ROOT / name / f'{name}_before_after_FL_ANG_deep_trajectory.png'
    fig.savefig(out_zoom, dpi=160)
    plt.close(fig)

    fig, axes = plt.subplots(2, 1, figsize=(13, 6.8), sharex=True)
    full_ylim = _local_ylim([base_fl], np.ones_like(frames, dtype=bool), pad_fraction=0.25)
    if full_ylim is None:
        full_ylim = (0.0, 120.0)
    axes[0].plot(frames, base_fl, color='tab:red', linewidth=1.5, label='baseline strict')
    for variant_name, values in variant_fl.items():
        _plot_series_with_outlier_markers(
            axes[0],
            frames,
            values,
            ylim=full_ylim,
            label=variant_name,
            color=variant_colors.get(variant_name, 'tab:blue'),
        )
    axes[0].set_ylim(*full_ylim)
    axes[0].set_ylabel(fl_label)
    axes[0].set_title('Full sequence sanity check: out-of-view markers are failed FL blow-ups, not corrected motion')

    axes[1].plot(frames, base_deep_mid, color='tab:red', linewidth=1.2, label='baseline filtered deep')
    axes[1].plot(frames, gated_deep_mid, color='tab:blue', linewidth=1.2, label='gated deep')
    axes[1].plot(frames, raw_deep_mid, color='0.6', linewidth=0.8, label='raw deep measurement')
    axes[1].scatter(np.flatnonzero(apo['line_rejected'][:, 1]), gated_deep_mid[apo['line_rejected'][:, 1]], color='black', s=12, label='deep rejects')
    axes[1].set_ylabel('deep mid-y (px)')
    axes[1].set_xlabel('frame')

    for ax in axes:
        ax.axvspan(min(problem) - 0.5, max(problem) + 0.5, color='gold', alpha=0.16, linewidth=0)
        ax.grid(True, alpha=0.25)
        ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    out_full = OUT_ROOT / name / f'{name}_full_sequence_sanity_check.png'
    fig.savefig(out_full, dpi=160)
    plt.close(fig)
    return [out_zoom, out_full]

plot_paths = [path for name in DATASETS for path in plot_variant_comparison(name)]
plot_paths

[PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/test_23june_2_before_after_FL_ANG_deep_trajectory.png'),
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/test_23june_2_full_sequence_sanity_check.png'),
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test23_june_3/test23_june_3_before_after_FL_ANG_deep_trajectory.png'),
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test23_june_3/test23_june_3_full_sequence_sanity_check.png')]

## 11. Problem-Frame Overlays

Overlay legend in these diagnostic images:

- red thick: baseline final Kalman fascicle;
- blue thick: `apo_gate` final Kalman fascicle;
- yellow thin: KLT propagated fascicle prior;
- green: baseline filtered deep aponeurosis;
- cyan: gated deep aponeurosis;
- magenta: raw deep-aponeurosis measurement when that frame was rejected.


In [13]:
def draw_problem_overlay(name: str, frame_idx: int) -> Path:
    spec = DATASETS[name]
    baseline = baselines[name]
    apo_gate = variants[name]['apo_gate']
    apo_log = variant_logs[name]['apo']
    meas = measurements[name]
    gray = read_gray_frame(spec['video'], frame_idx)
    vis = cv2.cvtColor(roi.ensure_uint8_image(gray), cv2.COLOR_GRAY2BGR)

    draw_line_1b(vis, baseline['klt_prior_segments'][frame_idx], (0, 220, 255), 1)
    draw_line_1b(vis, baseline['deep_apo_lines'][frame_idx], (0, 180, 0), 2)
    draw_line_1b(vis, apo_log['deep_lines'][frame_idx], (255, 255, 0), 2)
    if apo_log['line_rejected'][frame_idx, 1]:
        draw_line_1b(vis, meas['deep_measurement_lines'][frame_idx], (255, 0, 255), 2)
    draw_line_1b(vis, baseline['fascicle_segments'][frame_idx], (0, 0, 255), 4)
    draw_line_1b(vis, apo_gate['fascicle_segments'][frame_idx], (255, 0, 0), 3)

    base_fl = baseline.get('FL_mm', baseline['FL_px'])[frame_idx]
    gate_fl = apo_gate.get('FL_mm', apo_gate['FL_px'])[frame_idx]
    draw_text(vis, [
        f'{name} frame {frame_idx}',
        f'baseline FL {base_fl:.2f}, ANG {baseline["ANG_deg"][frame_idx]:.2f}',
        f'gated   FL {gate_fl:.2f}, ANG {apo_gate["ANG_deg"][frame_idx]:.2f}',
        f'deep reject: {bool(apo_log["line_rejected"][frame_idx, 1])}',
    ])
    out = OUT_ROOT / name / 'overlays' / f'{name}_frame_{frame_idx:06d}_baseline_vs_gated.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out), vis)
    return out

all_overlay_paths = []
for name, spec in DATASETS.items():
    for frame in spec['overlay_frames']:
        if frame < len(baselines[name]['frame']):
            all_overlay_paths.append(draw_problem_overlay(name, frame))
all_overlay_paths[:5], len(all_overlay_paths)


([PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/overlays/test_23june_2_frame_000102_baseline_vs_gated.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/overlays/test_23june_2_frame_000106_baseline_vs_gated.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/overlays/test_23june_2_frame_000107_baseline_vs_gated.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/overlays/test_23june_2_frame_000108_baseline_vs_gated.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/overlays/test_23june_2_frame_000110_baseline_vs_gated.png')],
 11)

## 12. Candidate-Line Debug Images

These images show the top saved Hough fascicle candidates on selected frames. The bold yellow line is the persistent-alpha selected candidate when it replaces the raw weighted-median alpha; otherwise the raw weighted-median stream is left unchanged.


In [14]:
def candidate_segment_from_xy(meas: Mapping[str, np.ndarray], frame_idx: int, peak_idx: int) -> np.ndarray:
    x = meas['candidate_x'][frame_idx, peak_idx]
    y = meas['candidate_y'][frame_idx, peak_idx]
    return np.asarray([x[0], y[0], x[1], y[1]], dtype=float)


def draw_candidate_debug(name: str, frame_idx: int) -> Path:
    spec = DATASETS[name]
    meas = measurements[name]
    alpha_log = variant_logs[name]['alpha_persist']
    gray = read_gray_frame(spec['video'], frame_idx)
    vis = cv2.cvtColor(roi.ensure_uint8_image(gray), cv2.COLOR_GRAY2BGR)
    palette = [(160, 160, 160), (120, 120, 255), (120, 200, 120), (200, 120, 120), (120, 200, 200)]
    for peak_idx in range(min(10, meas['candidate_x'].shape[1])):
        seg = candidate_segment_from_xy(meas, frame_idx, peak_idx)
        if not np.all(np.isfinite(seg)):
            continue
        draw_line_1b(vis, seg, palette[peak_idx % len(palette)], 1)
    selected_idx = int(alpha_log['selected_peak_idx'][frame_idx])
    if selected_idx >= 0:
        draw_line_1b(vis, candidate_segment_from_xy(meas, frame_idx, selected_idx), (0, 255, 255), 4)
    draw_text(vis, [
        f'{name} frame {frame_idx}',
        f'raw alpha {meas["alpha"][frame_idx]:.2f}',
        f'persistent alpha {alpha_log["alpha"][frame_idx]:.2f}',
        f'raw rejected: {bool(alpha_log["raw_rejected"][frame_idx])}',
    ])
    out = OUT_ROOT / name / 'candidate_debug' / f'{name}_frame_{frame_idx:06d}_fascicle_candidates.png'
    out.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out), vis)
    return out

candidate_debug_paths = []
for name, spec in DATASETS.items():
    frames = sorted(set(spec['overlay_frames'] + list(np.flatnonzero(variant_logs[name]['alpha_persist']['raw_rejected'])[:8])))
    for frame in frames:
        if frame < len(baselines[name]['frame']):
            candidate_debug_paths.append(draw_candidate_debug(name, frame))

candidate_debug_paths[:5], len(candidate_debug_paths)


([PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/candidate_debug/test_23june_2_frame_000035_fascicle_candidates.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/candidate_debug/test_23june_2_frame_000036_fascicle_candidates.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/candidate_debug/test_23june_2_frame_000037_fascicle_candidates.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/candidate_debug/test_23june_2_frame_000038_fascicle_candidates.png'),
  PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/candidate_debug/test_23june_2_frame_000059_fascicle_candidates.png')],
 27)

## 13. Local Window Tables


In [15]:
def local_window_table(name: str, frames: Sequence[int]) -> pd.DataFrame:
    baseline = baselines[name]
    apo_gate = variants[name]['apo_gate']
    apo_alpha = variants[name]['apo_gate_alpha_persist']
    apo_log = variant_logs[name]['apo']
    alpha_log = variant_logs[name]['alpha_persist']
    rows = []
    for frame in frames:
        if frame >= len(baseline['frame']):
            continue
        rows.append({
            'dataset': name,
            'frame': frame,
            'baseline_FL': float(baseline.get('FL_mm', baseline['FL_px'])[frame]),
            'apo_gate_FL': float(apo_gate.get('FL_mm', apo_gate['FL_px'])[frame]),
            'apo_alpha_FL': float(apo_alpha.get('FL_mm', apo_alpha['FL_px'])[frame]),
            'baseline_ANG': float(baseline['ANG_deg'][frame]),
            'apo_gate_ANG': float(apo_gate['ANG_deg'][frame]),
            'apo_alpha_ANG': float(apo_alpha['ANG_deg'][frame]),
            'raw_alpha': float(measurements[name]['alpha'][frame]),
            'persistent_alpha': float(alpha_log['alpha'][frame]),
            'deep_mid_baseline': float(line_mid_y(baseline['deep_apo_lines'])[frame]),
            'deep_mid_gated': float(line_mid_y(apo_log['deep_lines'])[frame]),
            'deep_rejected': bool(apo_log['line_rejected'][frame, 1]),
            'alpha_rejected': bool(alpha_log['raw_rejected'][frame]),
        })
    return pd.DataFrame(rows)

local_tables = []
for name, spec in DATASETS.items():
    lo = max(0, min(spec['problem_frames']) - 4)
    hi = min(len(baselines[name]['frame']) - 1, max(spec['problem_frames']) + 5)
    table = local_window_table(name, range(lo, hi + 1))
    table.to_csv(OUT_ROOT / name / f'{name}_problem_window_before_after.csv', index=False)
    local_tables.append(table)

pd.concat(local_tables, ignore_index=True)


,dataset,frame,baseline_FL,apo_gate_FL,apo_alpha_FL,baseline_ANG,apo_gate_ANG,apo_alpha_ANG,raw_alpha,persistent_alpha,deep_mid_baseline,deep_mid_gated,deep_rejected,alpha_rejected
0,test_23june_2,98,83.124960,84.655566,86.409334,25.681454,25.681454,25.489472,15.5,15.5,419.789811,441.704208,True,False
1,test_23june_2,99,83.440630,84.920648,86.682373,25.670857,25.670857,25.478862,15.5,15.5,419.472714,441.628143,True,False
2,test_23june_2,100,83.646696,85.175153,86.946651,25.645544,25.645544,25.453458,11.5,11.5,418.744727,441.288544,True,False
3,test_23june_2,101,84.206127,85.280381,87.056046,25.620774,25.620774,25.428601,16.0,16.0,419.149195,441.271591,True,False
4,test_23june_2,102,82.718542,84.904318,86.674580,25.719744,25.719744,25.526334,11.5,11.5,418.947485,441.734283,True,False
5,test_23june_2,103,81.768304,84.446195,86.203898,25.795757,25.795757,25.601602,14.0,14.0,418.872710,441.842010,True,False
6,test_23june_2,104,82.349869,84.803310,86.579696,25.764728,25.764728,25.570427,21.5,21.5,419.127863,441.979691,True,False
7,test_23june_2,105,85.436052,85.892598,87.746081,25.653760,25.653760,25.457353,21.5,21.5,420.882190,442.289825,True,False
8,test_23june_2,106,90.448772,87.709829,89.822092,25.393049,25.393049,25.180954,11.0,23.0,424.080667,443.006546,True,True
9,test_23june_2,107,95.635300,86.929773,89.068466,25.488007,25.488007,25.274614,18.0,18.0,430.437462,444.725174,True,False


## 14. Interpretation

Use this section after execution:

- If `apo_gate` reduces the FL spike and deep-mid jump inside the highlighted windows while keeping small outside-window median and p95 FL/ANG deltas, the production change should be an aponeurosis measurement gate plus per-endpoint R scaling.
- If it reduces the highlighted windows but fails the p95/rejection-rate checks, keep it as a diagnostic prototype and add a better reacquisition rule before changing the runner.
- If `apo_gate_alpha_persist` helps steep-angle periods without large outside-window suppression, expose fascicle angle limits and candidate-persistence parameters. If it suppresses plausible steep motion, keep it debug-only for now.
- The preferred production shape is configurable, not hard-coded: thresholds should become CLI/config options next to `--kalman-mode`, `--apo-maxangle`, and seed selection controls.

In [16]:
conclusion_rows = []
for _, row in metrics_df.iterrows():
    n_frames = len(baselines[row['dataset']]['frame'])
    reject_fraction = float(row['deep_line_rejects_total']) / max(n_frames, 1)
    improves_fl_step = row['problem_max_abs_FL_step'] < row['baseline_problem_max_abs_FL_step']
    improves_deep = row['problem_max_abs_deep_mid_step_px'] < row['baseline_problem_max_abs_deep_mid_step_px']
    preserves_ang_median = row['outside_median_abs_ANG_delta_vs_baseline'] < 1.0
    preserves_fl_median = row['outside_median_abs_FL_delta_vs_baseline'] < 2.0
    preserves_ang_p95 = row['outside_p95_abs_ANG_delta_vs_baseline'] < 2.0
    preserves_fl_p95 = row['outside_p95_abs_FL_delta_vs_baseline'] < 15.0
    reject_rate_ok = reject_fraction < 0.35
    conclusion_rows.append({
        'dataset': row['dataset'],
        'variant': row['variant'],
        'reduces_problem_FL_step': bool(improves_fl_step),
        'reduces_problem_deep_jump': bool(improves_deep),
        'preserves_outside_ANG_median_lt_1deg': bool(preserves_ang_median),
        'preserves_outside_FL_median_lt_2': bool(preserves_fl_median),
        'preserves_outside_ANG_p95_lt_2deg': bool(preserves_ang_p95),
        'preserves_outside_FL_p95_lt_15': bool(preserves_fl_p95),
        'deep_reject_fraction': reject_fraction,
        'deep_reject_fraction_lt_0_35': bool(reject_rate_ok),
        'recommended_for_main_code': bool(
            improves_fl_step
            and improves_deep
            and preserves_ang_median
            and preserves_fl_median
            and preserves_ang_p95
            and preserves_fl_p95
            and reject_rate_ok
        ),
    })
conclusion_df = pd.DataFrame(conclusion_rows)
conclusion_df.to_csv(OUT_ROOT / 'recommendation_matrix.csv', index=False)
print('wrote:', metrics_path)
print('wrote:', OUT_ROOT / 'recommendation_matrix.csv')
print('plots:')
for path in plot_paths:
    print(' ', path)
print('overlays:', len(all_overlay_paths))
print('candidate debug images:', len(candidate_debug_paths))
conclusion_df

wrote: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/variant_metrics.csv
wrote: /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/recommendation_matrix.csv
plots:
  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/test_23june_2_before_after_FL_ANG_deep_trajectory.png
  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test_23june_2/test_23june_2_full_sequence_sanity_check.png
  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test23_june_3/test23_june_3_before_after_FL_ANG_deep_trajectory.png
  /Users/grosbedou/PycharmProjects/NDORMS/results/notebook67_anatomical_gating_diagnostics/test23_june_3/test23_june_3_full_sequence_sanity_check.png
overlays: 11
candidate debug images: 27


,dataset,variant,reduces_problem_FL_step,reduces_problem_deep_jump,preserves_outside_ANG_median_lt_1deg,preserves_outside_FL_median_lt_2,preserves_outside_ANG_p95_lt_2deg,preserves_outside_FL_p95_lt_15,deep_reject_fraction,deep_reject_fraction_lt_0_35,recommended_for_main_code
0,test_23june_2,apo_gate,True,True,True,True,True,True,0.274112,True,True
1,test_23june_2,apo_gate_alpha_persist,True,True,True,False,True,True,0.274112,True,False
2,test23_june_3,apo_gate,True,True,True,True,True,False,0.355072,False,False
3,test23_june_3,apo_gate_alpha_persist,True,True,True,False,True,False,0.355072,False,False
